# Plot Checkpoint Predictions - v1

Select a trained checkpoint, one session date, and a ticker list. The notebook rebuilds chronological 1-minute windows from the provider bars, runs one-step inference, and plots predicted close vs target close for each ticker.

In [ ]:
from pathlib import Path

MODEL_VERSION = "v1"
MODEL_ROOT = Path(r"D:\\TradingData\\quant-research-workbench\\market_data\\models\\inhouse_transformer") / MODEL_VERSION

# Paste a trained v1 checkpoint path, or leave empty to auto-load the latest v1 last.pt/best.pt.
CHECKPOINT_PATH = ""

# Inference selection.
SESSION_DATE = "2024-01-22"
TICKERS = ("VFC", "USO", "CADL")

# Runtime controls.
WORKSPACE = Path(r"D:\\TradingCodes\\quant-research-workbench")
DEVICE = "cuda"
USE_AMP = True
INFERENCE_BATCH_SIZE = 2048

# 0 means plot every valid chronological window for each ticker.
MAX_POINTS_PER_TICKER = 0

In [ ]:
import sys
from dataclasses import fields
from datetime import datetime

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from research.inhouse_transformer.v1.config import DataConfig, ExperimentConfig, ModelConfig, TrainConfig
from research.inhouse_transformer.v1.data import available_sessions
from research.inhouse_transformer.v1 import train as train_mod

train_mod.load_torch_stack()

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (18, 8),
        "axes.titlesize": 15,
        "axes.labelsize": 12,
        "legend.fontsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "lines.linewidth": 2.0,
    }
)

np.set_printoptions(precision=5, suppress=True)
torch.set_printoptions(precision=5, sci_mode=False)

In [ ]:
def dataclass_from_dict(cls, payload, tuple_fields=()):
    allowed = {field.name for field in fields(cls)}
    values = {key: value for key, value in payload.items() if key in allowed}
    if cls is DataConfig and "processed_root" in values:
        values["processed_root"] = Path(values["processed_root"])
    for name in tuple_fields:
        if name in values:
            values[name] = tuple(values[name])
    return cls(**values)


def latest_checkpoint(model_root):
    candidates = list(model_root.glob("*/last.pt")) + list(model_root.glob("*/best.pt"))
    if not candidates:
        raise FileNotFoundError(f"No checkpoints found under {model_root}. Set CHECKPOINT_PATH manually.")
    return max(candidates, key=lambda path: path.stat().st_mtime)

checkpoint_path = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else latest_checkpoint(MODEL_ROOT)
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
if not TICKERS:
    raise ValueError("Set TICKERS to at least one ticker symbol.")

device = torch.device(DEVICE if DEVICE == "cpu" or torch.cuda.is_available() else "cpu")
checkpoint = torch.load(checkpoint_path, map_location="cpu")
checkpoint_config = checkpoint.get("config")
if not checkpoint_config:
    raise KeyError("Checkpoint does not contain a config payload, so model/data shape cannot be restored safely.")
checkpoint_version = checkpoint_config.get("experiment_version")
if checkpoint_version and checkpoint_version != MODEL_VERSION:
    raise ValueError(f"Checkpoint version {checkpoint_version!r} does not match notebook version {MODEL_VERSION!r}.")

data_config = dataclass_from_dict(
    DataConfig,
    checkpoint_config.get("data", {}),
    tuple_fields=("target_columns", "input_feature_columns", "time_feature_columns", "tickers"),
)
model_config = dataclass_from_dict(ModelConfig, checkpoint_config.get("model", {}))
train_config = dataclass_from_dict(TrainConfig, checkpoint_config.get("train", {}))

# Keep checkpoint model/data shape, but override the plotted session/tickers/runtime.
data_config.train_start_date = SESSION_DATE
data_config.train_end_date = SESSION_DATE
data_config.validation_start_date = SESSION_DATE
data_config.validation_end_date = SESSION_DATE
data_config.test_start_date = SESSION_DATE
data_config.test_end_date = SESSION_DATE
data_config.tickers = tuple(ticker.upper() for ticker in TICKERS)
data_config.max_tickers = len(data_config.tickers)
train_config.batch_size = INFERENCE_BATCH_SIZE
train_config.amp = USE_AMP

config = ExperimentConfig(data=data_config, model=model_config, train=train_config)

if "close" not in config.data.target_columns:
    raise ValueError(f"This notebook plots close, but checkpoint target_columns={config.data.target_columns}")

sessions = available_sessions(config.data.processed_root, SESSION_DATE, SESSION_DATE)
print(f"checkpoint={checkpoint_path}")
print(f"checkpoint_step={checkpoint.get('step')} best_score={checkpoint.get('best_score')}")
print(f"device={device} amp={config.train.amp and device.type == 'cuda'} batch_size={config.train.batch_size}")
print(f"session={sessions[0]} tickers={config.data.tickers}")
print(f"context={config.data.context_length} horizon={config.data.horizon} targets={config.data.target_columns}")
print(f"target_mode={config.data.target_mode} input_normalization={config.data.input_normalization}")
print(f"features={len(config.data.input_feature_columns)} time_features={len(config.data.time_feature_columns)}")

In [ ]:
model = train_mod.FeatureTemporalTransformer(
    feature_count=len(config.data.input_feature_columns),
    time_feature_count=len(config.data.time_feature_columns),
    context_length=config.data.context_length,
    horizon=config.data.horizon,
    target_count=len(config.data.target_columns),
    config=config.model,
).to(device)
model.load_state_dict(checkpoint["model"], strict=True)
model.eval()

param_count = sum(parameter.numel() for parameter in model.parameters())
print(f"loaded model parameters={param_count:,}")

In [ ]:
# Model architecture diagram with tensor shapes.
import math
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch


def _active_data_config():
    if "data_config" in globals():
        return data_config
    if "config" in globals() and hasattr(config, "data"):
        return config.data
    raise RuntimeError("Run the config-loading cells before drawing the architecture diagram.")


def _active_model_config():
    if "model_config" in globals():
        return model_config
    if "config" in globals() and hasattr(config, "model"):
        return config.model
    raise RuntimeError("Run the model-config cells before drawing the architecture diagram.")


def _tensor_shape(name, fallback):
    current_batch = globals().get("batch")
    if isinstance(current_batch, dict) and name in current_batch:
        return tuple(current_batch[name].shape)
    return fallback


def _box(ax, x, y, w, h, text, *, fc="#f7f9fc", ec="#365f91", fontsize=9):
    patch = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.02,rounding_size=0.018",
        linewidth=1.2,
        edgecolor=ec,
        facecolor=fc,
    )
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fontsize, wrap=True)
    return patch


def _arrow(ax, start, end):
    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=12,
            linewidth=1.0,
            color="#333333",
            shrinkA=4,
            shrinkB=4,
        )
    )


def _single_branch_diagram(data_cfg, model_cfg):
    feature_count = len(data_cfg.input_feature_columns)
    time_columns = tuple(getattr(data_cfg, "time_feature_columns", ()))
    time_count = len(time_columns)
    context_length = data_cfg.context_length
    horizon = data_cfg.horizon
    target_count = len(data_cfg.target_columns)
    values_shape = _tensor_shape("values", ("B", context_length, feature_count))
    time_shape = _tensor_shape("time_features", ("B", context_length, time_count)) if time_count else None

    fig, ax = plt.subplots(figsize=(18, 7))
    ax.set_axis_off()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    fig.suptitle(f"{MODEL_VERSION} Feature-Temporal Transformer", fontsize=16, fontweight="bold")

    _box(ax, 0.03, 0.62, 0.18, 0.16, f"values\n{values_shape}")
    if time_shape is not None:
        _box(ax, 0.03, 0.38, 0.18, 0.16, f"time_features\n{time_shape}", fc="#f3fbf6", ec="#3b7f4c")
        embed_text = "value projection + feature embedding\n+ position embedding + time projection"
        _arrow(ax, (0.21, 0.70), (0.28, 0.58))
        _arrow(ax, (0.21, 0.46), (0.28, 0.58))
    else:
        embed_text = "value projection + feature embedding\n+ position embedding\ntime already encoded in value channels"
        _arrow(ax, (0.21, 0.70), (0.28, 0.58))

    _box(ax, 0.28, 0.48, 0.18, 0.20, embed_text, fc="#fff8ed", ec="#b7791f")
    _box(
        ax,
        0.51,
        0.61,
        0.17,
        0.15,
        "feature Transformer\nattention across features\n"
        f"layers={model_cfg.feature_attention_layers}",
        fc="#eef6ff",
        ec="#2b6cb0",
    )
    _box(ax, 0.51, 0.41, 0.17, 0.12, "softmax feature pooling\nper bar -> [B, context, d_model]", fc="#f7f7ff", ec="#5a67d8")
    _box(
        ax,
        0.73,
        0.50,
        0.17,
        0.18,
        "temporal Transformer\nattention across bars\n"
        f"layers={model_cfg.temporal_layers}",
        fc="#eefaf7",
        ec="#2c7a7b",
    )
    _box(ax, 0.73, 0.25, 0.17, 0.12, "last token + LayerNorm\n[B, d_model]", fc="#f5f0ff", ec="#6b46c1")
    _box(ax, 0.93, 0.57, 0.06, 0.12, f"regression\n[B,{horizon},{target_count}]")
    _box(ax, 0.93, 0.33, 0.06, 0.12, f"direction\n[B,{horizon}]", fc="#fff5f5", ec="#c53030")

    _arrow(ax, (0.46, 0.58), (0.51, 0.68))
    _arrow(ax, (0.595, 0.61), (0.595, 0.53))
    _arrow(ax, (0.68, 0.47), (0.73, 0.59))
    _arrow(ax, (0.815, 0.50), (0.815, 0.37))
    _arrow(ax, (0.90, 0.31), (0.93, 0.39))
    _arrow(ax, (0.90, 0.31), (0.93, 0.63))

    ax.text(
        0.03,
        0.08,
        f"d_model={model_cfg.d_model} | heads={model_cfg.num_heads} | ff_dim={model_cfg.ff_dim} | "
        f"features={feature_count} | time_features={time_count}",
        fontsize=10,
        ha="left",
        va="center",
    )
    plt.show()


def _multiscale_diagram(data_cfg, model_cfg):
    feature_count = len(data_cfg.input_feature_columns)
    time_count = len(data_cfg.time_feature_columns)
    five_len = math.ceil(data_cfg.five_min_lookback_minutes / data_cfg.five_min_bucket_minutes)
    thirty_len = math.ceil(data_cfg.thirty_min_lookback_minutes / data_cfg.thirty_min_bucket_minutes)
    multiscale_feature_count = len(globals().get("MULTISCALE_FEATURE_COLUMNS", getattr(globals().get("train_mod", object()), "MULTISCALE_FEATURE_COLUMNS", tuple(range(15)))))
    anchor_feature_count = len(globals().get("ANCHOR_VALUE_COLUMNS", getattr(globals().get("train_mod", object()), "ANCHOR_VALUE_COLUMNS", tuple(range(11)))))
    anchor_count = len(globals().get("ANCHOR_NAMES", getattr(globals().get("train_mod", object()), "ANCHOR_NAMES", tuple(range(19)))))
    horizon = data_cfg.horizon
    target_count = len(data_cfg.target_columns)

    branches = [
        ("1m", _tensor_shape("values", ("B", data_cfg.context_length, feature_count)), _tensor_shape("time_features", ("B", data_cfg.context_length, time_count)), "last 64 1m bars"),
        ("5m", _tensor_shape("five_min_values", ("B", five_len, multiscale_feature_count)), _tensor_shape("five_min_time_features", ("B", five_len, time_count)), "5h aggregated into 5m buckets"),
        ("30m", _tensor_shape("thirty_min_values", ("B", thirty_len, multiscale_feature_count)), _tensor_shape("thirty_min_time_features", ("B", thirty_len, time_count)), "24h aggregated into 30m buckets"),
        ("anchors", _tensor_shape("anchor_values", ("B", anchor_count, anchor_feature_count)), _tensor_shape("anchor_time_features", ("B", anchor_count, time_count)), "same-minute + day/week summary anchors"),
    ]

    fig, ax = plt.subplots(figsize=(20, 10))
    ax.set_axis_off()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    fig.suptitle(f"{MODEL_VERSION} Multiscale Feature-Temporal Transformer", fontsize=16, fontweight="bold")

    y_positions = [0.78, 0.58, 0.38, 0.18]
    summary_points = []
    for (name, value_shape, time_shape, description), y in zip(branches, y_positions):
        _box(ax, 0.02, y, 0.12, 0.12, f"{name} values\n{value_shape}")
        _box(ax, 0.02, y - 0.12, 0.12, 0.09, f"time\n{time_shape}", fc="#f3fbf6", ec="#3b7f4c")
        _box(ax, 0.18, y - 0.04, 0.18, 0.12, f"{name} embedding\nvalue + feature + position + time\n{description}", fc="#fff8ed", ec="#b7791f", fontsize=8)
        _box(ax, 0.40, y - 0.04, 0.16, 0.12, f"feature Transformer\nfeatures inside token\nlayers={model_cfg.feature_attention_layers}", fc="#eef6ff", ec="#2b6cb0", fontsize=8)
        _box(ax, 0.60, y - 0.04, 0.15, 0.12, "feature pool\nthen temporal Transformer\n" f"layers={model_cfg.temporal_layers}", fc="#eefaf7", ec="#2c7a7b", fontsize=8)
        _box(ax, 0.79, y - 0.02, 0.10, 0.08, f"{name} summary\n[B,d_model]", fc="#f5f0ff", ec="#6b46c1", fontsize=8)
        _arrow(ax, (0.14, y + 0.04), (0.18, y + 0.02))
        _arrow(ax, (0.14, y - 0.075), (0.18, y + 0.02))
        _arrow(ax, (0.36, y + 0.02), (0.40, y + 0.02))
        _arrow(ax, (0.56, y + 0.02), (0.60, y + 0.02))
        _arrow(ax, (0.75, y + 0.02), (0.79, y + 0.02))
        summary_points.append((0.89, y + 0.02))

    _box(ax, 0.91, 0.55, 0.08, 0.16, "stack 4 summaries\n+ scale embedding\n[B,4,d_model]", fc="#fffaf0", ec="#b7791f", fontsize=8)
    _box(ax, 0.91, 0.35, 0.08, 0.13, "fusion Transformer\n1 layer\nmean pool + LN", fc="#edf2ff", ec="#5a67d8", fontsize=8)
    _box(ax, 0.91, 0.17, 0.08, 0.10, f"regression\n[B,{horizon},{target_count}]", fontsize=8)
    _box(ax, 0.91, 0.04, 0.08, 0.10, f"direction\n[B,{horizon}]", fc="#fff5f5", ec="#c53030", fontsize=8)
    for point in summary_points:
        _arrow(ax, point, (0.91, 0.63))
    _arrow(ax, (0.95, 0.55), (0.95, 0.48))
    _arrow(ax, (0.95, 0.35), (0.95, 0.27))
    _arrow(ax, (0.95, 0.35), (0.95, 0.14))

    ax.text(
        0.02,
        0.03,
        f"d_model={model_cfg.d_model} | heads={model_cfg.num_heads} | ff_dim={model_cfg.ff_dim} | "
        f"time_features={time_count} | branch encoders do not share weights",
        fontsize=10,
        ha="left",
        va="center",
    )
    plt.show()


def draw_model_architecture():
    data_cfg = _active_data_config()
    model_cfg = _active_model_config()
    if hasattr(data_cfg, "five_min_lookback_minutes"):
        _multiscale_diagram(data_cfg, model_cfg)
    else:
        _single_branch_diagram(data_cfg, model_cfg)


draw_model_architecture()


In [ ]:
rows_by_ticker = train_mod.infer_session_timeline_predictions(
    model=model,
    config=config,
    session=sessions[0],
    tickers=config.data.tickers,
    device=device,
    max_points_per_ticker=MAX_POINTS_PER_TICKER,
)

if not rows_by_ticker:
    raise RuntimeError("No valid prediction rows were created. Check date, tickers, context length, and horizon.")

summary_rows = []
for ticker, rows in rows_by_ticker.items():
    target_bps = np.array([float(row["target_bps"]) for row in rows])
    prediction_bps = np.array([float(row["prediction_bps"]) for row in rows])
    target_close = np.array([float(row["target_close"]) for row in rows])
    prediction_close = np.array([float(row["prediction_close"]) for row in rows])
    error_bps = prediction_bps - target_bps
    direction_correct = (prediction_bps > 0.0) == (target_bps > 0.0)
    summary_rows.append(
        {
            "ticker": ticker,
            "windows": len(rows),
            "target_start": rows[0]["target_time"],
            "target_end": rows[-1]["target_time"],
            "mae_price": float(np.mean(np.abs(prediction_close - target_close))),
            "mae_bps": float(np.mean(np.abs(error_bps))),
            "rmse_bps": float(np.sqrt(np.mean(error_bps ** 2))),
            "bias_bps": float(np.mean(error_bps)),
            "direction_accuracy_pct": float(np.mean(direction_correct) * 100.0),
        }
    )

summary = pl.DataFrame(summary_rows)
summary

In [ ]:
def plot_ticker_predictions(ticker, rows):
    rows = sorted(rows, key=lambda row: int(row["bar_index"]))
    times = [datetime.fromisoformat(str(row["target_time"])) for row in rows]
    target_close = np.array([float(row["target_close"]) for row in rows])
    prediction_close = np.array([float(row["prediction_close"]) for row in rows])
    target_bps = np.array([float(row["target_bps"]) for row in rows])
    prediction_bps = np.array([float(row["prediction_bps"]) for row in rows])
    error_bps = prediction_bps - target_bps
    direction_correct = (prediction_bps > 0.0) == (target_bps > 0.0)

    mae_bps = float(np.mean(np.abs(error_bps)))
    rmse_bps = float(np.sqrt(np.mean(error_bps ** 2)))
    bias_bps = float(np.mean(error_bps))
    dir_acc = float(np.mean(direction_correct) * 100.0)

    fig, (price_ax, error_ax) = plt.subplots(
        2,
        1,
        figsize=(18, 8),
        sharex=True,
        gridspec_kw={"height_ratios": [3.0, 1.1], "hspace": 0.08},
    )
    fig.suptitle(
        f"{ticker} | {SESSION_DATE} | h1 close prediction vs target | "
        f"MAE {mae_bps:.2f} bps | RMSE {rmse_bps:.2f} bps | bias {bias_bps:.2f} bps | dir {dir_acc:.1f}%",
        y=0.98,
        fontweight="bold",
    )

    price_ax.plot(times, target_close, color="#111827", label="target close", linewidth=2.3)
    price_ax.plot(
        times,
        prediction_close,
        color="#f97416c7",
        label="prediction close",
        linewidth=1.0,

    )
    price_ax.set_ylabel("close price")
    price_ax.legend(loc="upper left", frameon=True)
    price_ax.grid(True, color="#e5e7eb", linewidth=0.8)

    error_ax.axhline(0.0, color="#374151", linewidth=1.0)
    error_ax.fill_between(times, error_bps, 0.0, color="#f97316", alpha=0.18)
    error_ax.plot(times, error_bps, color="#ea580c", linewidth=1.3, label="prediction error bps")
    error_ax.set_ylabel("error bps")
    error_ax.set_xlabel("target bar time")
    error_ax.grid(True, color="#e5e7eb", linewidth=0.8)

    locator = mdates.AutoDateLocator(minticks=5, maxticks=12)
    formatter = mdates.ConciseDateFormatter(locator)
    error_ax.xaxis.set_major_locator(locator)
    error_ax.xaxis.set_major_formatter(formatter)
    fig.autofmt_xdate()
    plt.show()


for ticker, rows in rows_by_ticker.items():
    plot_ticker_predictions(ticker, rows)

In [ ]:
# Optional: inspect the exact plotted rows as a table.
all_rows = []
for ticker, rows in rows_by_ticker.items():
    for row in rows:
        all_rows.append(row)

pl.DataFrame(all_rows).select(
    "ticker",
    "bar_index",
    "target_time",
    "current_close",
    "target_close",
    "prediction_close",
    "target_bps",
    "prediction_bps",
    "abs_error_bps",
).head(50)